# 02_clean_orders_day02 — Bronze → Silver (functions & decorators)

**Week 2 · Pipeline Day 2** — Refactor Day 01’s loop into **named functions** and wrap the job with a **decorator**. Same silver result; cleaner structure for real pipelines.

**How to use this notebook**
- Run cells **top to bottom** (`Shift + Enter`).
- Read **§1** before coding or running the solution cells.


---
## §0. Scenario

Same six bronze rows as Day 01. Today you **do not** change the business rules — you change **how the code is organized**:

- Row filters → `is_analytics_ready(row)`
- City cleanup → `normalize_order_row(row)`
- Batch transform → `clean_orders(rows)`
- Orchestration → `@log_pipeline_run` on `run_pipeline()`


---
## §1. Your task (acceptance criteria)

### A. Functions (same rules as Day 01)

| Function | Behavior |
|----------|----------|
| `is_analytics_ready(row)` | `False` if `status != "completed"` or `amount is None` |
| `normalize_order_row(row)` | New dict with `city = row["city"].strip().upper()` |
| `clean_orders(rows)` | List of normalized rows where `is_analytics_ready` is true |

### B. Lambda (metrics only)

In `report_metrics`, find the **top order by `amount`** using `max(cleaned, key=lambda r: r["amount"])` and print one summary line.

### C. Decorator

Implement `log_pipeline_run` so it prints `[pipeline] start clean_orders_day02` before the job and `[pipeline] done clean_orders_day02` after (use `try` / `finally`).

Apply it with `@log_pipeline_run` on `run_pipeline()`.

### When correct, you should see:

```text
[pipeline] start clean_orders_day02
Pipeline Day 2 | valid_orders=3 | unique_users=3 | revenue=485.50
Users seen: ['u1', 'u2', 'u3']
Top order by amount → order_id=6, amount=310.0
[pipeline] done clean_orders_day02
```

Silver rows: `order_id` **1**, **4**, **6** with cities **HYD**, **BLR**, **DELHI**.


---
## §2. Input data (do not edit)


In [ ]:
RAW_ORDERS = [
    {"order_id": 1, "user_id": "u1", "status": "completed", "amount": 120.0, "city": " hyd "},
    {"order_id": 2, "user_id": "u2", "status": "pending", "amount": 90.0, "city": "blr"},
    {"order_id": 3, "user_id": "u1", "status": "completed", "amount": None, "city": " Chennai "},
    {"order_id": 4, "user_id": "u3", "status": "completed", "amount": 55.5, "city": "blr"},
    {"order_id": 5, "user_id": "u4", "status": "cancelled", "amount": 200.0, "city": "hyd"},
    {"order_id": 6, "user_id": "u2", "status": "completed", "amount": 310.0, "city": " delhi "},
]

print(f"bronze rows loaded: {len(RAW_ORDERS)}")


---
## §3. Solution — functions

**Instructor / self-check:** Run this cell or type the same logic yourself in a new cell above before peeking.


In [ ]:
import functools


def is_analytics_ready(row):
    if row.get("status") != "completed":
        return False
    if row.get("amount") is None:
        return False
    return True


def normalize_order_row(row):
    return {**row, "city": row["city"].strip().upper()}


def clean_orders(rows):
    return [normalize_order_row(r) for r in rows if is_analytics_ready(r)]


cleaned = clean_orders(RAW_ORDERS)
print(f"after clean_orders → {len(cleaned)} silver rows")
for row in cleaned:
    print(row)


---
## §4. Solution — decorator + metrics + run


In [ ]:
def report_metrics(cleaned):
    unique_users = {r["user_id"] for r in cleaned}
    top_order = max(cleaned, key=lambda r: r["amount"])
    total_revenue = sum(r["amount"] for r in cleaned)

    print(
        f"Pipeline Day 2 | valid_orders={len(cleaned)} "
        f"| unique_users={len(unique_users)} "
        f"| revenue={total_revenue:.2f}"
    )
    print(f"Users seen: {sorted(unique_users)}")
    print(
        f"Top order by amount → order_id={top_order['order_id']}, "
        f"amount={top_order['amount']}"
    )


def log_pipeline_run(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        print("[pipeline] start clean_orders_day02")
        try:
            return fn(*args, **kwargs)
        finally:
            print("[pipeline] done clean_orders_day02")
    return wrapper


@log_pipeline_run
def run_pipeline():
    cleaned = clean_orders(RAW_ORDERS)
    report_metrics(cleaned)


run_pipeline()
